In [ ]:
# ruff: noqa: ANN001, PTH123

# ############################################################################
# MODULE:      start_process
# AUTHOR(S):   Jonas Pischke, Victoria-Leandra Brunn, Julia Haas
#
# PURPOSE:     wrapper script to set variables and execute processing
#
# SPDX-FileCopyrightText: (c) 2025 by mundialis GmbH & Co. KG
#
# SPDX-License-Identifier: GPL-3.0-or-later
#
############################################################################

In [3]:
import json
import time
from pathlib import Path

import requests
from eodag import EODataAccessGateway
from requests.auth import HTTPBasicAuth


Define variables:

In [4]:
GRASS_PROJECT = "athen_urban-green_epsg32635"
START = "2025-12-01"
END = "2025-12-05"
TILE_ID = "35SKC"
MAIN_PC_PATH = "templates/"
MAIN_PROCESS_CHAIN = "pc_loop_ids.json"

Define actinia variables:

In [5]:
# variables to set the actinia host, version, and user
actinia_baseurl = "http://localhost:8088"
actinia_version = "v3"
actinia_url = actinia_baseurl + "/api/" + actinia_version
actinia_auth = HTTPBasicAuth("actinia", "actinia")  # username, password

### Filter Sentinel-2 scenes

In [6]:
search_criteria = {
    "productType": "S2MSI2A",
    "start": START,
    "end": END,
    "tileIdentifier": TILE_ID,
}

In [7]:
dag = EODataAccessGateway()
all_products = dag.search_all(**search_criteria)
s2_ids = [all_products[i].properties["id"] for i in range(len(all_products))]
print(f"Found <{len(s2_ids)}> Sentinel-2 scene IDs:")
for sid in s2_ids:
    print(f" - {sid}")

Found <2> Sentinel-2 scene IDs:
 - S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209
 - S2C_MSIL2A_20251203T092351_N0511_R093_T35SKC_20251203T130213


### Start actinia

In [8]:
# helper function to print formatted JSON using the json module
class HasBeenTerminatedError(Exception):
    """Throw exception class."""

    def __init__(self, request_url) -> None:
        """Throw exception."""
        super().__init__(f"The resource <{request_url}> has been terminated.")


def print_as_json(data) -> None:
    """Print request as json."""
    print(json.dumps(data, indent=2))


# helper function to verify a request
def verify_request(request, success_code=200) -> None:
    """Verify the request."""
    if request.status_code != success_code:
        print(
            "ERROR: actinia processing failed with status code "
            f"{request.status_code}!",
        )
        print("See errors below:")
        print_as_json(request.json())
        request_url = request.json()["urls"]["status"]
        requests.delete(url=request_url, auth=actinia_auth, timeout=20)
        raise HasBeenTerminatedError(request_url)

Construct actinia endpoint:
`...locations/{GRASS_PROJECT}/mapsets/PERMANENT/processing` -> persistent, data is kept, needs
`...locations/{GRASS_PROJECT}/processing_export` -> non-persistent, data is not kept in GRASSDB

In [16]:
# make a POST request to the actinia data API
request_url = f"{actinia_url}/locations/{GRASS_PROJECT}/processing_export"

print("actinia POST request:")
print(request_url)
print("---")

actinia POST request:
http://localhost:8088/api/v3/locations/athen_urban-green_epsg32635/processing_export
---


In [17]:
# load the main process chain
with open(MAIN_PC_PATH + MAIN_PROCESS_CHAIN, encoding="utf-8") as file:
    process_chain = json.load(file)

# insert Sentinel-2 IDs into the process chain
process_chain["list"][0]["inputs"][0]["value"] = s2_ids

# make the POST request to start the processing
request = requests.post(
    url=request_url, auth=actinia_auth, json=process_chain, timeout=20,
)
# check if anything went wrong
verify_request(request, 200)
# get a json-encoded content of the response
json_response = request.json()

# get status request URL
status_request_url = json_response["urls"]["status"]

# keep polling the status until finished
while json_response["status"] in {"accepted", "running"}:
    status_request = requests.get(
        url=status_request_url.replace("https", "http"), auth=actinia_auth,
        timeout=20,
    )
    json_response = status_request.json()
    print(f"Polling status for {s2_ids}: {json_response['status']}")
    print(f"Doing: {json_response['message']}")
    print("#########################################")
    time.sleep(30)

print(f"Final status for {s2_ids}: {json_response['status']}")
print(f"Processing for {s2_ids} finished.")
print("=========================================")

Polling status for ['S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209', 'S2C_MSIL2A_20251203T092351_N0511_R093_T35SKC_20251203T130213']: accepted
Doing: Resource accepted
#########################################
Polling status for ['S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209', 'S2C_MSIL2A_20251203T092351_N0511_R093_T35SKC_20251203T130213']: running
Doing: Running executable i.s2_id.download with parameters ['s2_id=S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209', 'download_dir=/home/data/s2'] for 30.174325942993164 seconds
#########################################
Polling status for ['S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209', 'S2C_MSIL2A_20251203T092351_N0511_R093_T35SKC_20251203T130213']: running
Doing: Running executable i.s2_id.download with parameters ['s2_id=S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209', 'download_dir=/home/data/s2'] for 60.29983854293823 seconds
######################################

In [15]:
status_request_url

'https://localhost:8088/api/v3/resources/actinia/resource_id-e1a4b92c-a92f-4625-97d9-a5faabb918bf'

Print links to tif of NDVI/NDWI results
TODO: Timestamp needs to be extracted from S2 ids and inserted into export names of NDVI/NDWI tifs in order get explicit names for the raster maps, e.g. `NDVI_20251206.tif` 

In [18]:
for tif in json_response["urls"]["resources"]:
    print(f"{Path(tif).name}: {tif.replace('https', 'http')}")

ndvi_S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209.tif: http://localhost:8088/api/v3/resources/actinia/resource_id-3ec51bd2-86a3-44dc-9387-6723f55ddb8f/ndvi_S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209.tif
ndvi_S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209_reclassified.tif: http://localhost:8088/api/v3/resources/actinia/resource_id-3ec51bd2-86a3-44dc-9387-6723f55ddb8f/ndvi_S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209_reclassified.tif
ndwi_S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209.tif: http://localhost:8088/api/v3/resources/actinia/resource_id-3ec51bd2-86a3-44dc-9387-6723f55ddb8f/ndwi_S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209.tif
ndwi_S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209_reclassified.tif: http://localhost:8088/api/v3/resources/actinia/resource_id-3ec51bd2-86a3-44dc-9387-6723f55ddb8f/ndwi_S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209_reclassified.tif
